In [0]:
CREATE OR REPLACE TABLE electronics_cat.gold.dim_customer AS
SELECT
    customerkey,
    gender,
    continent,
    country,
    state,
    city
FROM electronics_cat.silver.customers;

In [0]:
CREATE OR REPLACE TABLE electronics_cat.gold.dim_product AS
SELECT
    productkey,
    product_name,
    brand,
    color,
    category,
    subcategory,
    unit_price_usd
FROM electronics_cat.silver.products;

CREATE OR REPLACE TABLE electronics_cat.gold.dim_store AS
SELECT
    store_key,
    country AS store_country,
    state,
    square_meters
FROM electronics_cat.silver.stores;

CREATE OR REPLACE TABLE electronics_cat.gold.dim_date AS
SELECT DISTINCT
    order_date AS date,
    YEAR(order_date) AS year,
    MONTH(order_date) AS month,
    DAY(order_date) AS day
FROM electronics_cat.silver.sales
WHERE order_date IS NOT NULL;

CREATE OR REPLACE TABLE electronics_cat.gold.dim_exchange_rate AS
SELECT
    date,
    currency,
    exchange
FROM electronics_cat.silver.exc_rate;

In [0]:
CREATE OR REPLACE TABLE electronics_cat.gold.fact_sales AS
SELECT
    s.order_number,
    s.line_item,

    s.order_date,
    d.year,
    d.month,

    s.customerkey,
    s.product_key,
    s.storekey,

    s.quantity,

    p.unit_price_usd,

    er.exchange,

    -------------------------------------------------------------
    ROUND(
        (s.quantity * p.unit_price_usd) / er.exchange,
        2
    ) AS revenue_usd,

    -------------------------------------------------------------
    DATEDIFF(s.delivery_date, s.order_date) AS delivery_days,

    -------------------------------------------------------------
    CASE 
        WHEN s.storekey = -1 THEN 'online'
        ELSE 'store'
    END AS channel,

    s.currency_code

FROM electronics_cat.silver.sales s
LEFT JOIN electronics_cat.gold.dim_product p
    ON s.product_key = p.productkey
LEFT JOIN electronics_cat.gold.dim_date d
    ON s.order_date = d.date
LEFT JOIN electronics_cat.gold.dim_exchange_rate er
    ON s.currency_code = er.currency
    AND s.order_date = er.date;